# RAG over Wikipedia — retrieval-augmented question answering

**Goal**: build a working Retrieval-Augmented Generation pipeline end to
end — embed a text corpus, index it in a local vector store, retrieve
relevant chunks for a question, and ground an LLM's answer in them —
while treating retrieval quality as something to *measure*, not assume.

**Dataset**: [`rag-datasets/rag-mini-wikipedia`](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia)
(CC-BY) — 3,200 Wikipedia passages + 918 question/answer pairs, purpose-built
for RAG experimentation so the focus stays on the pipeline rather than data
collection.

**Stack**: Hugging Face (dataset + sentence-transformers) · FAISS ·
LangChain · a pluggable LLM backend (local Hugging Face model by default;
Ollama or OpenAI as drop-in alternatives).


In [1]:
import sys
import time

import pandas as pd
import plotly.express as px

sys.path.insert(0, "../src")
from rag_pipeline import (
    answer_question,
    build_vector_store,
    chunk_documents,
    evaluate_retrieval,
    get_llm,
    load_corpus,
    load_qa,
    passages_to_documents,
)


/Users/sarankansivananthan/rag-wikipedia/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load the data

In [2]:
corpus_df = load_corpus("../data/passages.parquet")
qa_df = load_qa("../data/qa_test.parquet")
print(f"{len(corpus_df)} passages, {len(qa_df)} question/answer pairs")
corpus_df.head(3)


3200 passages, 918 question/answer pairs


,passage
id,
0,"Uruguay (official full name in ; pron. , Eas..."
1,"It is bordered by Brazil to the north, by Arge..."
2,Montevideo was founded by the Spanish in the e...


In [3]:
qa_df.head(5)


,question,answer
id,,
0,Was Abraham Lincoln the sixteenth President of...,yes
2,Did Lincoln sign the National Banking Act of 1...,yes
4,Did his mother die of pneumonia?,no
6,How many long was Lincoln's formal education?,18 months
8,When did Lincoln begin his political career?,1832


Nearly half the gold answers are just "yes"/"no" — fine for the dataset's
original QA-accuracy purpose, but useless for evaluating *retrieval*
(checking whether "yes" appears in a retrieved chunk says nothing). The
evaluation step later filters these out.


In [4]:
print(qa_df["answer"].str.lower().str.strip(".").value_counts().head(5))
print(f"\n{(qa_df['answer'].str.lower().str.strip('.').isin(['yes', 'no'])).mean():.0%} of answers are yes/no")


answer
yes                     333
no                       75
john adams                3
the federalist party      3
grant                     3
Name: count, dtype: Int64

44% of answers are yes/no


## 2. Chunking strategy

The dataset's passages are already split at a reasonable size, but a real
corpus wouldn't be — so we re-chunk them with LangChain's
`RecursiveCharacterTextSplitter` at a few different sizes and *measure*
which one retrieves best, rather than picking one by feel.


In [5]:
documents = passages_to_documents(corpus_df)
print(f"{len(documents)} source documents")


3200 source documents


In [6]:
configs = [(100, 0), (250, 25), (500, 50), (1000, 100)]
chunking_results = []

for chunk_size, overlap in configs:
    chunks = chunk_documents(documents, chunk_size=chunk_size, chunk_overlap=overlap)
    t0 = time.time()
    vs = build_vector_store(chunks)
    build_time = time.time() - t0
    hit_rate, _ = evaluate_retrieval(vs, qa_df, k=4, sample_size=150)
    chunking_results.append({
        "chunk_size": chunk_size, "overlap": overlap,
        "n_chunks": len(chunks), "hit_rate": hit_rate, "build_time_s": build_time,
    })
    print(f"chunk_size={chunk_size:>5} overlap={overlap:>4}: {len(chunks):>5} chunks, "
          f"hit_rate={hit_rate:.1%}, build_time={build_time:.0f}s")

chunking_df = pd.DataFrame(chunking_results)


chunk_size=  100 overlap=   0: 14416 chunks, hit_rate=25.3%, build_time=95s


chunk_size=  250 overlap=  25:  7050 chunks, hit_rate=44.0%, build_time=97s


chunk_size=  500 overlap=  50:  4492 chunks, hit_rate=41.3%, build_time=98s


chunk_size= 1000 overlap= 100:  3405 chunks, hit_rate=42.7%, build_time=101s


In [7]:
fig = px.bar(
    chunking_df, x="chunk_size", y="hit_rate", text="n_chunks",
    labels={"chunk_size": "Chunk size (characters)", "hit_rate": "Retrieval hit rate @k=4"},
    title="Retrieval quality by chunk size (label = number of resulting chunks)",
)
fig.update_traces(texttemplate="%{text} chunks", textposition="outside")
fig.update_layout(yaxis_tickformat=".0%")
fig.show()


**Chunk size 100 is a clear loser** (25% hit rate vs. ~42-44% for larger
sizes) — chunks that small routinely split a fact away from the context
that would make it retrievable. Sizes 250-1000 are roughly comparable
here (a side effect of this dataset's passages already being fairly
short and self-contained); **250** edges out the others slightly and is
used for the rest of this notebook. On a real, less pre-cleaned corpus
this gap would likely be more pronounced — which is exactly the kind of
silent failure mode that motivates measuring retrieval instead of
assuming it works.


In [8]:
CHUNK_SIZE, CHUNK_OVERLAP = 250, 25
chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"{len(chunks)} chunks")


7050 chunks


## 3. Embeddings + FAISS index

Each chunk is embedded with a Hugging Face sentence-transformer
(`all-MiniLM-L6-v2`, 384 dimensions) and indexed in a local FAISS store —
no external database, no network calls at query time.


In [9]:
t0 = time.time()
vector_store = build_vector_store(chunks)
print(f"Indexed {len(chunks)} chunks in {time.time() - t0:.0f}s")

vector_store.save_local("../data/faiss_index")
print("Saved to ../data/faiss_index")


Indexed 7050 chunks in 104s
Saved to ../data/faiss_index


## 4. Retrieval evaluation

Before trusting any generated answer, we check something more basic:
**does retrieval even surface the right information?** For a sample of
questions (excluding yes/no answers, per the note in section 1), we
check whether the gold answer text appears in *any* of the top-k
retrieved chunks.


In [10]:
hit_rate, eval_results = evaluate_retrieval(vector_store, qa_df, k=4, sample_size=150)
print(f"Retrieval hit rate @k=4: {hit_rate:.1%}")
eval_results.head(10)


Retrieval hit rate @k=4: 44.0%


,question,answer,hit
0,Can citizens propose changes to the Constitution?,Yes. People are allowed to challenge laws.,False
1,Why are side-necked turtles called side-necked...,Because of the way they withdraw their heads i...,False
2,Where is Uruguay's oldest church?,"San Carlos, Maldonado.",True
3,Who was Roosevelt's presidential hero?,Lincoln was Roosevelt's presidential hero.,False
4,What includes a lengthy justification of why t...,Database entry.,False
5,Why did Coolidge not attend law school?,It was too expensive,False
6,Where do sea otters live?,Sea otters (Enhydra lutris) live along the Pac...,True
7,What happened in 1867?,"Canada was formed as a federal, semi-autonomou...",True
8,Where was Woodrow Wilson born?,"Woodrow Wilson was born in Staunton, Virginia",True
9,Who spend half of their life on land and half ...,Penguins,False


~44% is an honest, unglamorous number — and a realistic one for exact
substring matching against Wikipedia trivia (a correct answer can be
retrieved yet phrased differently than the dataset's gold string, which
this simple check would count as a miss). It's a deliberately strict
lower bound, not a final verdict: good enough to flag *when retrieval
quality changes* (e.g. across chunk sizes, embedding models), which is
its actual purpose here.


In [11]:
print("Examples where retrieval MISSED the answer:")
eval_results[~eval_results["hit"]].head(5)


Examples where retrieval MISSED the answer:


,question,answer,hit
0,Can citizens propose changes to the Constitution?,Yes. People are allowed to challenge laws.,False
1,Why are side-necked turtles called side-necked...,Because of the way they withdraw their heads i...,False
3,Who was Roosevelt's presidential hero?,Lincoln was Roosevelt's presidential hero.,False
4,What includes a lengthy justification of why t...,Database entry.,False
5,Why did Coolidge not attend law school?,It was too expensive,False


## 5. Generation, grounded in retrieved context

The LLM only sees the retrieved chunks — it's instructed to say
*"I don't know"* rather than guess when the context doesn't contain the
answer. `get_llm()` supports three interchangeable backends; the local
one needs no setup so it's what runs here:

```python
get_llm(provider="local")                       # Hugging Face model, in-process (used below)
get_llm(provider="ollama", model_name="mistral") # local server via Ollama
get_llm(provider="openai", model_name="gpt-4o-mini")  # hosted, needs OPENAI_API_KEY
```


In [12]:
llm = get_llm(provider="local")


Device set to use cpu


In [13]:
questions = [
    "Was Abraham Lincoln the sixteenth President of the United States?",
    "What is the capital of Uruguay?",
    "Who discovered penicillin?",
]

for q in questions:
    answer, docs = answer_question(vector_store, llm, q, k=4)
    print(f"Q: {q}")
    print(f"A: {answer}")
    for rank, d in enumerate(docs):
        print(f"   [{rank}] {d.page_content[:140]}...")
    print()


Q: Was Abraham Lincoln the sixteenth President of the United States?
A: yes
   [0] Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until hi...
   [1] Young Abraham Lincoln...
   [2] On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinrid...
   [3] Scholars now rank Lincoln among the top three U.S. Presidents, with the majority of those surveyed placing him first. He is noted for his la...



Q: What is the capital of Uruguay?
A: Montevideo
   [0] Montevideo, Uruguay's capital....
   [1] Map of Uruguay...
   [2] Uruguay's capital, Montevideo, was founded by the Spanish in the early 18th century as a military stronghold; its natural harbor soon develo...
   [3] Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It...



Q: Who discovered penicillin?
A: I don't know
   [0] also wrote in defense of the scientific method....
   [1] As a chemist, Faraday discovered benzene, investigated the clathrate hydrate of chlorine, invented an early form of the bunsen burner and th...
   [2] great contribution to chemistry was recognised and he is recognised as a great Italian chemist. (Note: In 1911, Victor Emmanuel III was the ...
   [3] and ensuring that no one else was able to practice it. As he states in a letter to Boulton of August 17 1784:...



The third question is the interesting one: **none of the top-4 retrieved
chunks are about Alexander Fleming** — the actual discoverer of
penicillin. Instead retrieval surfaces a chunk about **Michael Faraday**
(who discovered *benzene*, not penicillin — clearly close enough in
embedding space, as a chemist's biography, to be retrieved, and far
enough in reality to be wrong). This is retrieval failing exactly the
way the intro describes: silently, with no error, no low-confidence
flag — just the wrong context handed to the model. Because the prompt
explicitly permits "I don't know," the model doesn't paper over the gap
with a plausible-sounding guess — but a less carefully prompted
pipeline, or a less careful eyeballing of outputs, would ship that
hallucination straight to a user. It's exactly this kind of failure that
section 4's evaluation step is designed to catch in aggregate rather
than one anecdote at a time.


## 6. Takeaways

- **Chunk size has a real, measurable effect on retrieval** — 100-character
  chunks retrieve the right context 25% of the time vs. ~42-44% for
  250-1000 characters. Not a detail to skip.
- **Retrieval quality should be measured, not assumed.** A generation
  step has no way to know its context is wrong — it will produce a
  fluent, confident, incorrect answer just as readily as a correct one.
  The `evaluate_retrieval` hit-rate check exists specifically to catch
  this before it reaches a user.
- **Grounding the prompt with an explicit "I don't know" escape hatch**
  turns a retrieval failure into a visible non-answer instead of a
  silent hallucination — cheap to add, and demonstrated directly in
  section 5.
- **The LLM backend is a swappable detail, not the core of the system**
  — `get_llm()` runs the exact same pipeline against a local
  Hugging Face model, a local Ollama/Mistral server, or the OpenAI API
  with a one-line change.
